# Tools
- Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code.
- Tools are pairings of:


    1. A schema, including the name of the tool, description, and / or argument definitions (often a JSON schema)
    2. A function or coroutine to execute

In [2]:
from langchain.chat_models import init_chat_model
import os

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model(model="groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response 

AIMessage(content='<think>\nOkay, so I need to figure out why parrots talk. Let me start by recalling what I know about parrots. I know they\'re part of the Psittaciformes order, right? They\'re known for their ability to mimic human speech. But why do they do that? Maybe it\'s related to their social behavior. I\'ve heard that parrots are social animals, living in flocks in the wild. So maybe they use vocalizations to communicate with each other. But when they\'re in captivity, they mimic humans. Is there a link between their natural communication and their ability to mimic human speech?\n\nI remember reading that some birds have a part of the brain called the syrinx, which helps them produce sounds. Parrots might have a more developed syrinx for complex sounds. Also, their vocal learning is different from humans. Humans learn speech through hearing and repeating, while parrots might have a different mechanism. But why would they evolve this ability? In the wild, maybe they use mimicr

In [5]:
from langchain.tools import tool


@tool
def get_weather(location: str) -> str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])


In [6]:
response = model_with_tools.invoke("What is the weather like in Boston?")
print(response)

for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': "Okay, the user is asking about the weather in Boston. I need to use the get_weather function. Let me check the function parameters. It requires a location, which is Boston here. I'll call the function with location set to Boston. Make sure the JSON is correctly formatted with the name and arguments.\n", 'tool_calls': [{'id': 'w88djanck', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 86, 'prompt_tokens': 154, 'total_tokens': 240, 'completion_time': 0.144514051, 'completion_tokens_details': {'reasoning_tokens': 62}, 'prompt_time': 0.00873822, 'prompt_tokens_details': None, 'queue_time': 0.070497156, 'total_time': 0.153252271}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_d58dbe76cd', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019f5cba-3857-7e50-b1

## Tool Execution Loops

In [ ]:
# Step 1: Model generates topol calls
messages = [{"role": "user", "content": "What is the weather in BOston?"}] 
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)


# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

#Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

The weather in Boston is sunny today. That means there should be no rain, and temperatures are likely warm. Perfect for outdoor activities!


In [8]:
messages

[{'role': 'user', 'content': 'What is the weather in BOston?'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. The user wrote "BOston" but I should correct that to "Boston" to make sure it\'s the right location. I\'ll call the get_weather function with location set to Boston. No other parameters needed. Just need to return the function call in the specified format.\n', 'tool_calls': [{'id': 'ax1dc9vp6', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 111, 'prompt_tokens': 154, 'total_tokens': 265, 'completion_time': 0.203213472, 'completion_tokens_details': {'reasoning_tokens': 87}, 'prompt_time': 0.006045367, 'prompt_tokens_details': None, 'queue_time': 0.103817888, 'total_time': 0.209258839}, 'model_name': '